In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# --- load the data (paths from .env, falling back to ../data) -----------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
load_dotenv(ROOT / ".env")
DATA_DIR = Path(os.getenv("DATA_DIR", ROOT / "data"))

train = pd.read_csv(os.getenv("TRAIN_CSV", DATA_DIR / "train.csv"))
test  = pd.read_csv(os.getenv("TEST_CSV",  DATA_DIR / "test.csv"))

# --- missing values, train vs test -------------------------------------------
missing = pd.DataFrame({
    "dtype":         train.dtypes.astype(str),
    "train_missing": train.isna().sum().astype("Int64"),
    "train_pct":     (100 * train.isna().mean()).round(2),
    "test_missing":  test.isna().sum().astype("Int64"),   # NaN for Survived: not in test
    "test_pct":      (100 * test.isna().mean()).round(2),
})
missing["total_missing"] = missing[["train_missing", "test_missing"]].sum(axis=1).astype("Int64")
missing = missing.sort_values("total_missing", ascending=False)

print(f"train: {train.shape[0]} rows x {train.shape[1]} cols     "
      f"test: {test.shape[0]} rows x {test.shape[1]} cols\n")
# print with "-" wherever the column simply does not exist in that file
print(missing.astype(object).where(missing.notna(), "-").to_string())
print(f"\ncolumns with missing values : {(missing['total_missing'] > 0).sum()} of {len(missing)}"
      f"  ->  {missing.index[missing['total_missing'] > 0].tolist()}")
print(f"rows fully complete         : train {train.dropna().shape[0]}/{len(train)}, "
      f"test {test.dropna().shape[0]}/{len(test)}")
print("(Survived is '-' in test because the column does not exist there — it is the target.)")



train: 891 rows x 12 cols     test: 418 rows x 11 cols

               dtype train_missing train_pct test_missing test_pct total_missing
Cabin            str           687      77.1          327    78.23          1014
Age          float64           177     19.87           86    20.57           263
Embarked         str             2      0.22            0      0.0             2
Fare         float64             0       0.0            1     0.24             1
Name             str             0       0.0            0      0.0             0
Parch          int64             0       0.0            0      0.0             0
PassengerId    int64             0       0.0            0      0.0             0
Pclass         int64             0       0.0            0      0.0             0
Sex              str             0       0.0            0      0.0             0
SibSp          int64             0       0.0            0      0.0             0
Survived       int64             0       0.0         

In [2]:
# --- outlier check on the numeric columns -------------------------------------
# IQR fence (Q1-1.5*IQR, Q3+1.5*IQR) and z-score (|z|>3), computed per file.
# PassengerId is an id and Pclass/Survived are categorical, so neither is tested.
NUM_COLS = ["Age", "Fare", "SibSp", "Parch"]

rows = []
for name, df in [("train", train), ("test", test)]:
    for c in NUM_COLS:
        s = df[c].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        z = (s - s.mean()) / s.std()
        n_iqr = int(((s < lo) | (s > hi)).sum())
        rows.append({
            "file": name, "col": c,
            "min": round(s.min(), 2), "max": round(s.max(), 2),
            "Q1": round(q1, 2), "Q3": round(q3, 2), "IQR": round(iqr, 2),
            "low_fence": round(lo, 2), "high_fence": round(hi, 2),
            "n_IQR_out": n_iqr, "pct_IQR": round(100 * n_iqr / len(s), 1),
            "n_z>3": int((z.abs() > 3).sum()),
            "skew": round(s.skew(), 2),
        })

outliers = pd.DataFrame(rows).set_index(["file", "col"])
print("IQR and z-score outlier counts\n")
print(outliers.to_string())

# the extreme values themselves, so we can judge whether they are errors
print("\n" + "-" * 70)
print("the actual extremes (train + test combined)\n")
full = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
for c in NUM_COLS:
    top = full.nlargest(3, c)[["PassengerId", "split", "Pclass", "Name", c]]
    print(f"  highest {c}:")
    for _, r in top.iterrows():
        print(f"     {r[c]:>8}  id={r['PassengerId']:<5} {r['split']:<5} class {r['Pclass']}  {r['Name'][:45]}")
low_age = [round(float(a), 2) for a in sorted(full["Age"].dropna().unique())[:5]]
print(f"\n  lowest Age: {low_age}  <- infants stored as fractions of a year")

# sanity checks that matter more than the fence
print("\n" + "-" * 70)
print("plausibility checks\n")
print(f"  negative values           : {int((full[NUM_COLS] < 0).sum().sum())}")
print(f"  Age > 100                 : {int((full['Age'] > 100).sum())}")
print(f"  Fare == 0                 : {int((full['Fare'] == 0).sum())}  <- free passage, real records")
print(f"  Fare == 512.33 (the max)  : {int((full['Fare'] == full['Fare'].max()).sum())} rows, "
      f"all on ticket {full.loc[full['Fare'].idxmax(), 'Ticket']} -> one group booking, not a typo")

print("\n  zero-inflation warning (the IQR rule misbehaves on count columns):")
for c in ["SibSp", "Parch"]:
    s_c = full[c]
    pct0 = 100 * (s_c == 0).mean()
    q1, q3 = s_c.quantile([0.25, 0.75])
    iqr = q3 - q1
    hi = q3 + 1.5 * iqr
    if iqr == 0:
        why = (f"Q1=Q3={q1:.0f} so IQR=0 and the fence collapses onto {hi:.0f} -> it flags "
               f"EVERY non-zero value. Verdict meaningless.")
    else:
        why = (f"Q1={q1:.0f}, Q3={q3:.0f}, so the fence is {hi:.1f} -> it flags anyone with "
               f">{hi:.1f} relatives aboard, which is a large family, not an error.")
    print(f"     {c}: {pct0:.1f}% of rows are 0. {why}")

print("\n  Fare judged within its own Pclass instead of against everyone:")
for p in sorted(full["Pclass"].unique()):
    s = full.loc[full["Pclass"] == p, "Fare"].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    hi = q3 + 1.5 * iqr
    print(f"     class {p}: median {s.median():6.2f}  fence {hi:7.2f}  "
          f"outliers {int((s > hi).sum()):3d}")

IQR and z-score outlier counts

              min     max     Q1    Q3    IQR  low_fence  high_fence  n_IQR_out  pct_IQR  n_z>3  skew
file  col                                                                                            
train Age    0.42   80.00  20.12  38.0  17.88      -6.69       64.81         11      1.5      2  0.39
      Fare   0.00  512.33   7.91  31.0  23.09     -26.72       65.63        116     13.0     20  4.79
      SibSp  0.00    8.00   0.00   1.0   1.00      -1.50        2.50         46      5.2     30  3.70
      Parch  0.00    6.00   0.00   0.0   0.00       0.00        0.00        213     23.9     15  2.75
test  Age    0.17   76.00  21.00  39.0  18.00      -6.00       66.00          2      0.6      1  0.46
      Fare   0.00  512.33   7.90  31.5  23.60     -27.51       66.91         55     13.2     18  3.69
      SibSp  0.00    8.00   0.00   1.0   1.00      -1.50        2.50         11      2.6      7  4.17
      Parch  0.00    9.00   0.00   0.0   0.00     